In [2]:
%pip install scikit-learn

  Using cached scikit_learn-1.9.1-cp314-cp314-win_amd64.whl.metadata (9.3 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached threadpoolctl-3.7.0-py3-none-any.whl.metadata (24 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
Using cached scikit_learn-1.9.1-cp314-cp314-win_amd64.whl (8.4 MB)
Using cached joblib-1.6.0-py3-none-any.whl (306 kB)
Using cached cloudpickle-3.1.2-py3-none-any.whl (22 kB)
Using cached threadpoolctl-3.7.0-py3-none-any.whl (26 kB)

   -------- ------------------------------- 1/5 [narwhals]
   -------- ------------------------------- 1/5 [narwhals]
   -------- ------------------------------- 1/5 [narwhals]
   -------- ------------------------------- 1/5 [narwhals]
   -------- ------------------------------- 1/5 [narwhals]
   -------- ------------------------------- 1/5 [narwhals]
   -------- ------------------------------- 1/5 [narwhals]
   -------- ------------------------------- 1/5 [narwhals]
   -------- ----


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Week 2: Regression, California housing
Goal: predict `median_house_value` (in dollars) for each district.
Order of work: Look, Split, Baseline, Model, Explain, So what.

## 1. Look first

In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

url = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"
df = pd.read_csv(url)

print("Rows and columns:", df.shape)
print()
print("Missing values (only columns with any):")
print(df.isna().sum()[df.isna().sum() > 0])
print()
print("Text column ocean_proximity:")
print(df["ocean_proximity"].value_counts())
print()
print("Largest house value:", df["median_house_value"].max())
print("Districts at that exact value:", (df["median_house_value"] == df["median_house_value"].max()).sum())

Rows and columns: (20640, 10)

Missing values (only columns with any):
total_bedrooms    207
dtype: int64

Text column ocean_proximity:
ocean_proximity
<1H OCEAN     9136
INLAND        6551
NEAR OCEAN    2658
NEAR BAY      2290
ISLAND           5
Name: count, dtype: int64

Largest house value: 500001.0
Districts at that exact value: 965


**Write:** The data has 20,640 districts and only total_bedrooms has missing values (207). The largest house value is 500,001 and 965 districts sit exactly there, so prices look capped at about $500k.

## 2. Split before anything else
Every step after this (filling missing values, encoding, the model) is fitted on the **training set only**.

In [4]:
X = df.drop(columns="median_house_value")
y = df["median_house_value"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train rows:", len(X_train), "| Test rows:", len(X_test))

Train rows: 16512 | Test rows: 4128


**Write:** I split the data into 16,512 training rows and 4,128 test rows, so the test rows stay hidden until the end.

## 3. Baseline: predict the training mean for every test district

In [5]:
train_mean = y_train.mean()
baseline_pred = np.full(len(y_test), train_mean)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
print("Training mean used for every guess: $", round(train_mean))
print("Baseline RMSE: $", round(baseline_rmse))

Training mean used for every guess: $ 207195
Baseline RMSE: $ 114486


**Write:** Guessing the same average price for every district gives an RMSE of about $114,486, so any real model must beat that number.

## 4. LinearRegression on all the columns
Missing values are filled with the training median. `ocean_proximity` is one hot encoded.

In [6]:
num_cols = [c for c in X_train.columns if c != "ocean_proximity"]
cat_cols = ["ocean_proximity"]

prep = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
model = Pipeline([("prep", prep), ("lr", LinearRegression())])
model.fit(X_train, y_train)          # fitted on TRAIN only

test_pred = model.predict(X_test)
train_pred = model.predict(X_train)

test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
test_mae = mean_absolute_error(y_test, test_pred)
test_r2 = r2_score(y_test, test_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))

print("Test  RMSE: $", round(test_rmse))
print("Test  MAE : $", round(test_mae))
print("Test  R2  :", round(test_r2, 3))
print("Train RMSE: $", round(train_rmse))

Test  RMSE: $ 70059
Test  MAE : $ 50670
Test  R2  : 0.625
Train RMSE: $ 68434


**Write:** On the test set the model has RMSE about $70,059, MAE about $50,670 and R2 0.625. Train RMSE is about $68,434, close to test RMSE, so the model is not overfitting.

## 5. The coefficient that matters most

In [7]:
names = model.named_steps["prep"].get_feature_names_out()
coefs = pd.Series(model.named_steps["lr"].coef_, index=names).sort_values()
print(coefs.round(1))
print()
print("median_income summary: min", df["median_income"].min(), "max", df["median_income"].max())

cat__ocean_proximity_INLAND        -58713.2
num__longitude                     -26838.3
num__latitude                      -25468.4
cat__ocean_proximity_NEAR BAY      -24063.2
cat__ocean_proximity_<1H OCEAN     -18926.6
cat__ocean_proximity_NEAR OCEAN    -15495.4
num__population                       -38.2
num__total_rooms                       -6.0
num__households                        48.3
num__total_bedrooms                   102.8
num__housing_median_age              1102.2
num__median_income                  39474.0
cat__ocean_proximity_ISLAND        117198.5
dtype: float64

median_income summary: min 0.4999 max 15.0001


**Write:** The coefficient for median_income is +39,474. One extra unit of median income (about $10,000 a year) adds about $39,474 to the predicted house value, with the other columns held the same. The dummy columns are bigger in size, but they are only comparisons with each other (and ISLAND has just 5 rows), so median_income is the cleanest one to explain.

## 6. So what

In [8]:
print("Average error (MAE): $", round(test_mae))
print("Average price in test set: $", round(y_test.mean()))
print("Error as share of average price:", round(100 * test_mae / y_test.mean()), "%")

Average error (MAE): $ 50670
Average price in test set: $ 205500
Error as share of average price: 25 %


**Write:** On average the model is about $50,670 off, which is about 25% of the average price of $205,500. I would not use it to price a single house, because that is too much error and the $500k cap hides the top end. I would use it for rough screening of districts.